# 教程 0：十分钟上手 PLLM

你熟悉 PyTorch，大概也用 vLLM 跑过推理。这篇教程要带你做一件这两样工具单独都做不到的事：**在 LLM 逐 token 生成的同时，跑一段你自己写的张量数学**——比如把某一层的 hidden states 导出到宿主机，让模型一边说话、你一边「看」它在想什么。

听起来像要去改引擎源码？不用。PLLM 的思路是把「做什么」和「怎么做」分开：你用类型标注**声明**要算什么数学、读哪些张量、在什么时机运行；框架把这份声明编译成一份由 Rust 签名的执行计划，再交给托管运行时照着执行。你写的那点数学代码里，不会出现一行调度、锁或 CUDA 流管理的代码。

整篇教程（乃至整个框架）只需要先记住三个词，我们会在用到它们的时候再展开：

1. **Workload** —— 一个 Python 类，用类型标注声明：*算什么数学、读哪些张量、用什么状态、在什么时机运行*。里面只有 PyTorch。
2. **签名编译** —— `make_llm` 把你的声明编译成由 Rust 签名的执行计划；形状、dtype、依赖、效果在这一步被一次性检查固定，没有「跑到一半才发现」的意外。
3. **托管运行时** —— 状态的版本、隔离、发布、回收全由运行时按计划执行；你的数学代码只负责算。

走完这篇，你会跑通「加载引擎 → 生成 → 挂上 workload 观测 → 读回结果」这条最基础的链路。后面五篇教程的每一个场景，都建立在这条链路上。

## 第一次生成：先让引擎说句话

装好 PLLM 之后，我们先做一件最小的事：让引擎说一句话。别小看这一步——`make_llm` 不传 workload 时就是一台普通的 vLLM 引擎，而加载的这几秒里，模型权重、KV cache 预算、批处理上限都已各就各位，后面所有教程都建立在这台引擎之上。

`ModelConfig` 的位置参数依次是：模型名、dtype、最大序列长度、最大并发请求数、单批 token 数、显存占比。我们故意把它们都压得很小（0.5B 模型、只取 0.2 的显存），因为教程机器上的 GPU 是共享的——这也是你平时试 PLLM 的好习惯：先小后大。

下面发两个 prompt，各要 8 个 token，并用 `temperature=0.0` 关掉采样随机性。跑完你应该看到两行续写：「天空是……」后面跟着一句英文，「水结冰于……」后面跟着 `0 degrees Celsius` 之类的文本。

In [1]:
# 注意日志里的 (EngineCore pid=...)：纯 vLLM 引擎（不带 workload）走 vLLM
# 自己的构造路径，默认会额外 spawn 一个 EngineCore 子进程，这是正常现象。
# 待会带 workload 的引擎不一样：pllm 的运行时在 multiprocessing=False
# （默认）下会把引擎留在本进程内，你不需要为此做任何事。

import pllm
from vllm import SamplingParams
from pllm.config import ModelConfig

llm = pllm.make_llm(config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", "bfloat16",
    64,   # max_model_len
    2,    # max_num_seqs
    128,  # max_num_batched_tokens
    0.2,  # gpu_memory_utilization
    seed=7))

outputs = llm.generate(
    ["The sky is", "Water freezes at"],
    SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True),
    use_tqdm=False)
for output in outputs:
    print(repr(output.outputs[0].text))

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 18:40:00,964	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 18:40:01 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


INFO 09-10 18:40:03 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:40:03 [model.py:1883] Using max model len 64


WARNING 09-10 18:40:03 [arg_utils.py:2620] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.


... (52 outputs left out)

(EngineCore pid=1815773) 

INFO 09-10 18:40:12 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 18:40:16 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


' a beautiful place. It is a place'
' 0 degrees Celsius and boils at '


两行输出对应两个 prompt，各续写了 8 个 token。由于 `temperature=0.0` 加上固定的 `seed=7`，你在自己机器上重跑应当看到一模一样的文本——如果不是，先检查是否动了这两个参数。

加载日志也值得扫一眼，其中有两行最能说明问题：一行告诉你 0.2 的显存预算最终换算成了多少 token 的 KV cache；另一行 `enforce_eager=True` 说明我们关掉了 CUDA graph 和 torch.compile。教程阶段全程用 eager 模式，图捕获与性能优化是部署时的事，现在不分心。

## 挂上你的第一个 workload

现在来做 PLLM 真正想让你做的事。假设你好奇：模型每生成一个 token 的时候，第 0 层的 hidden states 长什么样？在普通 vLLM 里，这通常意味着改 forward、加钩子、自己搬张量；在 PLLM 里，你只需要写一个类——就是下面这个。

你可能会问：框架怎么知道什么时候调用 `observe`？`hidden` 这个参数又是谁喂进来的？答案全写在标注里，读这个类只要认得三个声明：

- `@pllm.on(pllm.Phase.OBSERVE)` 声明**时机**：采样前的观测窗口，引擎每走一步就触发一次；
- 参数上的 `Annotated[torch.Tensor, pllm.Hidden(0)]` 声明**输入来源**：第 0 层的 hidden states，框架负责把它作为实参递进来；
- `@pllm.export(...)` 声明**效果**：把返回值有界地复制到宿主机的 trace。`max_rows`/`max_cols` 就是这个「界」——导出的尺寸有硬上限，trace 永远不会悄悄撑爆内存。

再看函数体：只有一行普普通通的 PyTorch。这就是 PLLM 和你之间的约定——**你写数学，框架管其余**。至于每个 Phase 具体卡在引擎的哪个位置、export 的完整语义，教程 1 会逐个拆开；现在先让它跑起来。

In [2]:
from typing import Annotated
import torch


class PeekHidden(pllm.Workload):
    """每步导出第 0 层 hidden 的行均值。"""

    @pllm.on(pllm.Phase.OBSERVE)
    @pllm.export(name="hidden_mean", every=1, max_rows=4, max_cols=896)
    def observe(self, hidden: Annotated[torch.Tensor, pllm.Hidden(0)]) -> torch.Tensor:
        return hidden.float().mean(dim=-1, keepdim=True)

## 编译、运行、读结果

把 workload 交给 `make_llm`，就是前面说的「签名编译」：你的声明在这一步被检查、固定并签名，之后的运行只是按计划执行。

下面的代码做三件事。先把刚才那台引擎关掉、让出显存（共享 GPU 上的基本礼仪）；再造一台带 workload 的新引擎，发同样两个 prompt；最后读两样东西——这也是你以后调试任何 workload 时的固定动作：

- `trace_snapshot()`：导出事件所在的有界 trace。每个事件带着 step_id、请求身份，以及一个已经搬到 CPU 的张量，随时可取；
- `worker_state().diagnostics`：执行面的体检报告。眼下只需盯住一个字段：`fallback_count == 0` 表示一切都在签名原语内运行，你的数学没有一行「掉出」计划之外。

我们预计会看到：每个 step 一条 `hidden_mean` 事件，每条事件里有两个数——因为我们发了两个请求，它们正并肩走过引擎的每一步。

In [3]:
import gc

# 先关掉上一台引擎，给带 workload 的新引擎让出显存
llm.llm_engine.engine_core.shutdown()
del llm
gc.collect()
torch.cuda.empty_cache()

peek = pllm.make_llm(workload=PeekHidden(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", "bfloat16", 64, 2, 128, 0.2, seed=7))
try:
    peek.generate(["The sky is", "Water freezes at"],
        SamplingParams(max_tokens=4, temperature=0.0, ignore_eos=True), use_tqdm=False)
    for event in peek.trace_snapshot().events:
        if event.tensor_name == "hidden_mean":
            print(f"step {event.step_id}: 请求 "
                  f"{[i.prompt_idx for i in event.identities]} 的均值 "
                  f"{event.tensor().flatten().tolist()}")
    print("体检报告 diagnostics:", peek.worker_state().diagnostics)
finally:
    peek.close()

INFO 09-10 18:40:16 [core_client.py:686] [shutdown] MPClient: start timeout=default


INFO 09-10 18:40:16 [core_client.py:688] [shutdown] MPClient: stopping engine manager


INFO 09-10 18:40:16 [utils.py:612] [shutdown] Process manager: send sigterm to process EngineCore


INFO 09-10 18:40:16 [core_client.py:690] [shutdown] MPClient: engine manager stopped


INFO 09-10 18:40:16 [core_client.py:691] [shutdown] MPClient: cleaning up background resources


INFO 09-10 18:40:16 [core_client.py:693] [shutdown] MPClient: complete


... (24 outputs left out)

<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 09-10 18:40:23 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


INFO 09-10 18:40:23 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.68 s


step 1: 请求 [0, 1] 的均值 [0.0001381508045597002, -0.0018154947319999337]
step 2: 请求 [0, 1] 的均值 [0.0024056904949247837, 0.006180447060614824]
step 3: 请求 [0, 1] 的均值 [0.0032387205865234137, 0.000969733577221632]
体检报告 diagnostics: StateDiagnostics(active_lineages=0, inference_pins=0, loss_count=0, update_count=0, trained_rows=0, completed_sampler_pins=3, pending_state_releases=0, fallback_count=0, state_usage=PageBundleUsage(reserved_bytes=0, logical_live_bytes=0, allocated_payload_bytes=0, reserved_padding_bytes=0, free_field_pages=0, live_field_pages=0, private_field_pages=0, retiring_field_pages=0, private_versions=0, retiring_versions=0, pinned_versions=0, pending_receipts=0, admission_capacity_failures=0))


来对一下预期：trace 里每个 step 一行、每行两个均值，对应两个请求在同一观测窗口下的行均值。这些数字本身没有什么戏剧性，重要的是你刚才做到的事——**引擎一步没停，你却拿到了它内部每一步、每个请求的中间量**，代价只是一个几行的类。

体检报告里的大部分字段现在是零，这很正常：我们还没有声明任何状态，自然没有版本、没有 pin、也没有训练行数。到教程 3、4，这些字段会一个个亮起来。今天只需确认一件事：`fallback_count=0`——你的数学全程运行在签名计划之内，框架没有悄悄替你降级执行过任何东西。

## 小结与下一步

回顾一下，这篇教程下来你已经建立了三个直觉：

- `make_llm(config=...)` 就是 vLLM；多传一个 `workload=`，就是 vLLM + 你的数学。
- workload 里只有 PyTorch；**时机、输入、效果**用标注声明，编译期签名，运行时托管。
- 结果从 trace 里读，健康状况看 diagnostics——而 `fallback_count` 永远应该是 0。

链路已经通了。从这里开始，教程分成两条支线，按你的角色选一条——当然，两条都读最好：

**做算法的你** → 继续主线教程 1–5，每篇给这条链路加上一种真实能力：

- [教程 1：观测与 SAE](01_observe_sae.ipynb) —— 把今天的「偷看」做正经：多层观测、冻结状态、非阻塞落盘，最后给 hidden states 过一道 SAE；
- [教程 2：写自己的训练头](02_custom_head.ipynb) —— 逐步蒸馏、采样引导、CNN；
- [教程 3：按请求路由的 LoRA](03_request_lora.ipynb) —— 请求隔离、route 共享、混合与调度；
- [教程 4：三种循环](04_loops.ipynb) —— 层循环、步循环、会话循环；
- [教程 5：decode 时在线训练](05_decode_training.ipynb) —— 跨步与跨请求的训练目标。

**做系统的你** → 转入内部机制支线，拆开这台引擎看它怎么工作（纯 CPU 就能跑完，不需要 GPU）：

- [内部机制 1：签名编译管线](internals/01_signed_compilation.ipynb) —— 你的声明如何一步步变成签名的执行计划；
- [内部机制 2：托管状态与页 arena](internals/02_state_arena.ipynb) —— 版本、COW、pin 与显存预算的纪律；
- [内部机制 3：进程边界](internals/03_process_boundary.ipynb) —— workload 如何按引用跨进程传递，两端如何复核签名。

两条支线在「契约」这一个概念上汇合：算法侧学会怎么写声明，系统侧看懂声明如何被执行和守护。随时可以从一边跳回另一边。